# Prediction, step by step

This notebook follows one prediction from the raw tables to the final scorecard. At every
step it shows which function is called, which arrays go in and come out, their shapes,
and what the numbers mean.

Each stage repeats the body of a library function line by line, prints every
intermediate object with `peek`, and then checks with an `assert` that the result equals
what the library itself returns. The walkthrough therefore cannot drift away from the
real code without failing.

| Stage | What happens | Library code |
| --- | --- | --- |
| A | raw tables to the features of the players at risk | `features.snapshot_features` |
| B | features to an `xgboost.DMatrix` | `SurvivalModel._matrix` |
| C | booster: margin, leaves, SHAP contributions | `predict_margin`, `shap_risk` |
| D | margin to a survival curve on the player's own clock, for Cox and for AFT | `predict_survival` |
| E | calibration of the curve | `IsotonicSurvivalCalibrator.transform` |
| F | curve to the scorecard | `pipeline.score_players` |
| G | check against what is possible by the churn definition | - |

Nothing is trained and nothing is written to disk: the notebook only reads `data/` and
`artifacts/`, so run `python -m churn.generate_data` and `python -m churn.pipeline` first.

In [1]:
import warnings

# Optuna imports tqdm.auto, which warns when ipywidgets is missing; no progress bar is used.
warnings.filterwarnings("ignore", message="IProgress not found")

In [2]:
import json
import sys
from collections.abc import Callable
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import xgboost as xgb
from IPython.display import display
from plotly.colors import qualitative
from scipy.special import expit, ndtr

from churn import config, features
from churn import evaluation as ev
from churn.models import MODELS, AFTModel, CoxModel, SurvivalModel
from churn.pipeline import MODEL_COLUMNS, score_players

SNAPSHOT = config.SCORING_SNAPSHOT  # any date inside the data works
N_DEBUG = 5  # players followed through the prediction

DATA_DIR = config.DATA_DIR
ARTIFACTS_DIR = config.ARTIFACTS_DIR
MODELS_DIR = ARTIFACTS_DIR / "models"
TIMES = config.TIME_GRID  # days 1..85 on which survival curves are evaluated
HORIZON = config.HORIZON_DAYS
FEATURES = list(config.FEATURES)

pd.set_option("display.precision", 4, "display.max_columns", 40, "display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

if not (ARTIFACTS_DIR / "metrics.json").exists() or not (DATA_DIR / "activity.parquet").exists():
    raise FileNotFoundError(
        "data/ or artifacts/ is missing; run `uv run python -m churn.generate_data` "
        "and `uv run python -m churn.pipeline` first"
    )

## Tools: `peek` and the call tracer

`peek` prints the type, shape, dtype, range and first values of an object, together with
what it means, and adds a row to a journal that is shown as one table at the end.

`trace_calls` runs a function under `sys.setprofile` and prints every call that lands in
the `churn` package, indented by depth, with the shapes of the arguments and of the
returned value.

In [3]:
JOURNAL: list[dict[str, str]] = []
PREVIEW_DAYS = (1, 7, 14, 30, 60, 85)  # days shown when a (players, days) matrix is previewed
stage = "setup"  # set at the top of every stage; `peek` reads it


def peek(name: str, value: object, meaning: str, on_grid: bool = False) -> None:
    """Print what ``value`` is and what it means, and record it in the journal.

    ``on_grid=True`` marks a (players, days) matrix: its preview shows the days in
    ``PREVIEW_DAYS`` instead of the first columns.
    """
    kind, shape, dtype, stats = type(value).__name__, "", "", ""
    if isinstance(value, pd.DataFrame):
        shape, dtype = str(value.shape), "mixed"
        preview = value.head(3).to_string()
    elif isinstance(value, pd.Series | pd.Index | np.ndarray):
        array = np.asarray(value)
        shape, dtype = str(array.shape), str(array.dtype)
        if array.dtype.kind == "b":
            stats = f"true={array.sum()} false={(~array).sum()}"
        elif array.dtype.kind in "iuf" and array.size:
            numbers = array.astype(float)
            low, high, missing = np.nanmin(numbers), np.nanmax(numbers), np.isnan(numbers).sum()
            stats = f"min={low:.6g} max={high:.6g} nan={missing}"
        if isinstance(value, pd.Series):
            preview = value.head(4).to_string()
        elif on_grid:
            corner = array[:5, np.searchsorted(TIMES, PREVIEW_DAYS)]
            preview = f"days {', '.join(map(str, PREVIEW_DAYS))} (first rows):\n"
            preview += np.array2string(corner, precision=4, suppress_small=True)
        else:
            corner = array[:5, :6] if array.ndim == 2 else array[:8]
            preview = np.array2string(corner, precision=4, suppress_small=True, max_line_width=100)
    else:
        preview = repr(value)
    header = " ".join(part for part in (kind, shape and f"shape={shape}", dtype, stats) if part)
    print(f"[{stage}] {name}: {header}")
    print(f"    meaning: {meaning}")
    print("    " + preview.replace("\n", "\n    "))
    row = {"stage": stage, "object": name, "type": kind, "shape": shape, "dtype": dtype}
    JOURNAL.append(row | {"meaning": meaning})


def summarise(value: object) -> str:
    """One-word description of an argument or a return value for the call trace."""
    if isinstance(value, np.ndarray | pd.DataFrame | pd.Series):
        return f"{type(value).__name__}{value.shape}"
    if isinstance(value, tuple):
        return "(" + ", ".join(summarise(item) for item in value) + ")"
    if isinstance(value, pd.Timestamp):
        return str(value.date())
    if isinstance(value, int | float | str | bool | None):
        return repr(value)
    return type(value).__name__


def trace_calls(function: Callable[..., Any], *args: Any) -> Any:
    """Run ``function(*args)`` and print the calls it makes inside the ``churn`` package."""
    package_dir = str(config.ROOT / "churn")
    lines: list[str] = []
    depth = 0

    def profiler(frame: Any, event: str, arg: Any) -> None:
        nonlocal depth
        code = frame.f_code
        if not code.co_filename.startswith(package_dir) or code.co_name.startswith("<"):
            return
        indent = "    " * depth
        if event == "call":
            names = code.co_varnames[: code.co_argcount + code.co_kwonlyargcount]
            local = frame.f_locals
            shown = ", ".join(
                f"{n}={summarise(local[n])}" for n in names if n not in ("self", "cls")
            )
            owner = f"{type(local['self']).__name__}." if "self" in local else ""
            lines.append(f"{indent}{owner}{code.co_name}({shown})")
            depth += 1
        elif event == "return":
            depth -= 1
            lines.append(f"{'    ' * depth}  -> {summarise(arg)}")

    previous = sys.getprofile()
    sys.setprofile(profiler)
    try:
        result = function(*args)
    finally:
        sys.setprofile(previous)
    print("\n".join(lines))
    return result

## Inputs: models, calibrators, data and the players to follow

- `production` is the model that scores real players: the selected family refitted on
  every snapshot before the scoring date. Which class it is (Cox or AFT) is recorded in
  its meta file.
- `calibrator` is its isotonic calibrator, taken from the model fitted three months
  earlier. The scores use it only if model selection chose the calibrated variant
  (`best_variant` in `metrics.json`); stage E shows what it does either way.
- Stage D needs one model of each kind, so the evaluation model of the other kind is
  loaded as well, with the calibrator of its own.

The followed players are chosen to cover the whole range of `recency_days`, the feature
the prediction depends on most.

In [4]:
def load_model(name: str) -> SurvivalModel:
    """Load a fitted model; its class is recorded in the meta file."""
    prefix = MODELS_DIR / name
    meta = json.loads(Path(f"{prefix}.meta.json").read_text())
    return MODELS[meta["model"]].load(prefix)


def load_calibrator(name: str) -> ev.IsotonicSurvivalCalibrator:
    """Load the calibrator saved as ``calibrator_<name>.json``."""
    data = json.loads((MODELS_DIR / f"calibrator_{name}.json").read_text())
    return ev.IsotonicSurvivalCalibrator.from_dict(data)


metrics = json.loads((ARTIFACTS_DIR / "metrics.json").read_text())
best, best_variant = metrics["best_model"], metrics["best_variant"]
production = load_model("production")
# The production model, which is of the best family, comes with its own calibrator.
calibrators = {name: load_calibrator("production" if name == best else name) for name in MODELS}
calibrator = calibrators[best]
scoring_calibrator = calibrator if best_variant == "calibrated" else None  # what the scores use
cox = production if isinstance(production, CoxModel) else load_model("cox")
aft = production if isinstance(production, AFTModel) else load_model("aft")
print(f"production model: {type(production).__name__}, {production.num_boost_round} trees")
print(f"calibrator: days {calibrator.times_[0]}..{calibrator.times_[-1]}; scores: {best_variant}")

players, activity = features.load_raw(DATA_DIR)
churn_dates = features.first_churn_dates(activity, config.DATA_END)
frame = features.snapshot_features(players, activity, churn_dates, SNAPSHOT)

# One player near each of N_DEBUG evenly spaced recency values (lowest id on ties).
by_id = frame.sort_values("player_id")
targets = np.linspace(by_id["recency_days"].min(), by_id["recency_days"].max(), N_DEBUG)
picked = [(by_id["recency_days"] - target).abs().idxmin() for target in targets]
debug = frame.loc[list(dict.fromkeys(picked))]
print(f"snapshot {SNAPSHOT.date()}: {len(frame):,} players at risk, following {len(debug)}")
display(debug[["player_id", "recency_days", "active_days_7d", "active_days_30d", "tenure_days"]])

production model: CoxModel, 275 trees
calibrator: days 1..85; scores: calibrated


snapshot 2026-07-01: 4,328 players at risk, following 5


,player_id,recency_days,active_days_7d,active_days_30d,tenure_days
0,2,1,6,26,207
21,121,8,0,9,512
260,1194,16,0,7,271
231,1064,23,0,2,269
87,438,30,0,1,67


## Call map

The big picture before the details: the two library calls that turn raw tables into a
scorecard, with everything they call inside the package. Arguments and results are shown
as `Type(shape)`.

In [5]:
print("1. Features of everybody at risk on the snapshot\n")
trace_calls(features.snapshot_features, players, activity, churn_dates, SNAPSHOT)
print("\n2. Scoring of the followed players\n")
_ = trace_calls(score_players, production, scoring_calibrator, debug)

1. Features of everybody at risk on the snapshot

snapshot_features(players=DataFrame(20000, 5), activity=DataFrame(1700671, 9), churn_dates=Series(15672,), snapshot=2026-07-01)
  -> DataFrame(4328, 19)

2. Scoring of the followed players

score_players(model=CoxModel, calibrator=IsotonicSurvivalCalibrator, scoring=DataFrame(5, 19))
    CoxModel.predict_survival(X=DataFrame(5, 16), times=ndarray(85,))
        CoxModel._delay(X=DataFrame(5, 16))
          -> ndarray(5,)
        CoxModel.predict_margin(X=DataFrame(5, 16))
            CoxModel._matrix(X=DataFrame(5, 16))
              -> DMatrix
          -> ndarray(5,)
        CoxModel._survival(margin=ndarray(5,), model_times=ndarray(5, 85))
            CoxModel.baseline_cumulative_hazard(times=ndarray(5, 85))
              -> ndarray(5, 85)
          -> ndarray(5, 85)
      -> ndarray(5, 85)
    IsotonicSurvivalCalibrator.transform(surv=ndarray(5, 85), times=ndarray(85,))
      -> ndarray(5, 85)
    CoxModel.shap_risk(X=DataFrame(5, 16

## Stage A. Raw tables to features

`features.snapshot_features` decides who is at risk on the snapshot date and aggregates
their recent activity. The steps below are the ones that matter for the logic: only
activity strictly before the snapshot is visible, and a player is at risk if they have
played before and their first churn has not happened yet.

In [6]:
stage = "A"
peek("players", players, "one row per player: registration date and static traits")
peek("activity", activity, "one row per player and active day")
peek("churn_dates", churn_dates, "date of the first churn by player_id; absent = never churned")

history = activity[activity["date"] < SNAPSHOT]
peek("history", history, "activity strictly before the snapshot: all the model may see")
last_active = history.groupby("player_id")["date"].max()
peek("last_active", last_active, "last active day of every player who has played")
churn = churn_dates.reindex(last_active.index)
still_at_risk = (churn.isna() | (churn >= SNAPSHOT)).to_numpy()
peek("still_at_risk", still_at_risk, "True if the first churn has not happened before the snapshot")
at_risk = last_active.index[still_at_risk]
peek("at_risk", at_risk, "player ids that get a row and a prediction")
window = history[history["date"] >= SNAPSHOT - pd.Timedelta(days=config.INACTIVITY_DAYS)]
peek("window", window, "activity of the last 30 days, the source of the *_30d features")

# Three features recomputed by hand and compared with the library's output.
recency = (SNAPSHOT - last_active.reindex(at_risk)).dt.days
active_30d = window.groupby("player_id").size().reindex(at_risk)
turnover_30d = window.groupby("player_id")["turnover"].sum().reindex(at_risk)
peek("recency_days", recency, "days since the last activity, 1 = played yesterday")
peek("frame", frame, "output of snapshot_features: ids, date, 15 features, delay, churn date")
delay_all = frame[config.DELAY_COLUMN]
peek(config.DELAY_COLUMN, delay_all, "days during which churn is impossible: 30 - recency_days")

assert frame["player_id"].tolist() == at_risk.tolist()
assert (delay_all.to_numpy() == config.INACTIVITY_DAYS - recency.to_numpy()).all()
assert (frame["recency_days"].to_numpy() == recency.to_numpy()).all()
assert (frame["active_days_30d"].to_numpy() == active_30d.to_numpy()).all()
assert np.allclose(frame["turnover_30d"], turnover_30d)
print("\nOK: the at-risk set and the recomputed features equal features.snapshot_features")

[A] players: DataFrame shape=(20000, 5) mixed
    meaning: one row per player: registration date and static traits
       player_id registration_date  age acquisition_channel favorite_game
    0          1        2024-03-22   23             organic   crash_games
    1          2        2025-12-06   50           affiliate         slots
    2          3        2025-08-19   41           affiliate         slots
[A] activity: DataFrame shape=(1700671, 9) mixed
    meaning: one row per player and active day
       player_id       date  sessions  bets  turnover    ggr  deposits  withdrawals  bonus
    0       2316 2024-01-01         1    73     33.61  16.18     20.93          0.0    0.0
    1       2442 2024-01-01         1    70     90.95  28.54    109.15          0.0    0.0
    2       3674 2024-01-01         1    42     19.73   0.79     16.66          0.0    0.0
[A] churn_dates: Series shape=(15672,) datetime64[us]
    meaning: date of the first churn by player_id; absent = never churned
 

## Stage B. Features to a DMatrix

The booster sees only the 15 feature columns. `churn_free_days` travels with them in `X`
but is not a feature: stage D uses it to set the time origin. XGBoost reads the two categorical columns as
integer codes and handles them natively (`enable_categorical=True`), so the categories
must be declared in the same order as in training; `snapshot_features` takes that order
from `config.CATEGORICAL_FEATURES`.

In [7]:
stage = "B"
X = debug[MODEL_COLUMNS]
peek("X", X, "features and the delay of the followed players, one row per player")
display(X.dtypes.astype(str).rename("dtype").to_frame())

codes = X[list(config.CATEGORICAL_FEATURES)].apply(lambda column: column.cat.codes)
peek("category codes", codes, "the integers XGBoost sees instead of the category names")

matrix = xgb.DMatrix(X[production.feature_names_], enable_categorical=True)
peek("matrix.num_row()", matrix.num_row(), "players in the DMatrix")
peek("matrix.num_col()", matrix.num_col(), "features in the DMatrix")
peek("matrix.feature_types", matrix.feature_types, "type flag per feature, 'c' = categorical")
assert matrix.feature_names == production.feature_names_
assert config.DELAY_COLUMN not in matrix.feature_names

[B] X: DataFrame shape=(5, 16) mixed
    meaning: features and the delay of the followed players, one row per player
         tenure_days  recency_days  active_days_7d  active_days_30d  activity_ratio_7d_30d  sessions_30d  turnover_30d  ggr_30d  deposits_30d  withdrawals_30d  bonus_30d  avg_stake_30d  age acquisition_channel favorite_game  churn_free_days
    0            207             1               6               26                  0.989            53       4318.95   407.72       1103.87            25.97       6.86         1.3812   50           affiliate         slots               29
    21           512             8               0                9                  0.000            20        841.02    99.47        124.86             6.24      26.86         0.6871   24            paid_ads         slots               22
    260          271            16               0                7                  0.000            15       1983.46    67.84        362.85            30.42  

,dtype
tenure_days,int64
recency_days,int64
active_days_7d,int64
active_days_30d,int64
activity_ratio_7d_30d,float64
sessions_30d,int64
turnover_30d,float64
ggr_30d,float64
deposits_30d,float64
withdrawals_30d,float64


[B] category codes: DataFrame shape=(5, 2) mixed
    meaning: the integers XGBoost sees instead of the category names
         acquisition_channel  favorite_game
    0                      1              0
    21                     2              0
    260                    1              2
[B] matrix.num_row(): int
    meaning: players in the DMatrix
    5
[B] matrix.num_col(): int
    meaning: features in the DMatrix
    15
[B] matrix.feature_types: list
    meaning: type flag per feature, 'c' = categorical
    ['int', 'int', 'int', 'int', 'float', 'int', 'float', 'float', 'float', 'float', 'float', 'float', 'int', 'c', 'c']


## Stage C. The booster

The booster is a sum of regression trees. Its raw output, the **margin**, is one number
per player:

- Cox: `eta(x)`, the log hazard ratio. Larger means earlier churn.
- AFT: `mu(x)`, the location of `log T`. Larger means later churn.

Both refer to the player's own clock, the time left once churn is possible (stage D).

Three more views of the same trees: the margin after the first `k` trees, the leaf each
player falls into in every tree, and the TreeSHAP split of the margin into one
contribution per feature plus a bias.

In [8]:
stage = "C"
margin_meaning = {
    "cox": "eta(x): log hazard ratio, larger = earlier churn",
    "aft": "mu(x): location of log(days to churn), larger = later churn",
}[production.name]
booster = production.booster_
peek("booster.num_boosted_rounds()", booster.num_boosted_rounds(), "trees in the model")

eta = booster.predict(matrix, output_margin=True).astype(float)
peek("margin", eta, margin_meaning)
assert np.allclose(eta, production.predict_margin(X))

transformed = booster.predict(matrix)
peek("booster.predict(matrix)", transformed, "without output_margin XGBoost returns exp(margin)")
assert np.allclose(transformed, np.exp(eta), rtol=1e-4)

checkpoints = [k for k in (1, 5, 10, 25, 50, 100, 200, 300) if k < production.num_boost_round]
checkpoints.append(production.num_boost_round)
by_trees = np.stack(
    [booster.predict(matrix, output_margin=True, iteration_range=(0, k)) for k in checkpoints]
)
peek("margin by number of trees", by_trees, "rows = checkpoints, columns = players")
assert np.allclose(by_trees[-1], eta)

leaves = booster.predict(matrix, pred_leaf=True)
peek("leaves", leaves, "leaf index of every player in every tree: (players, trees)")

contributions = booster.predict(matrix, pred_contribs=True)
peek("contributions", contributions, "TreeSHAP: one column per feature, bias in the last one")
assert np.allclose(contributions.sum(axis=1), eta, rtol=1e-4, atol=1e-3)
shap = production.risk_sign * contributions[:, :-1].astype(float)
peek("shap (risk-oriented)", shap, "contributions signed so that positive = pushes towards churn")
assert np.allclose(shap, production.shap_risk(X))
print("\nOK: margin, exp(margin), the sum of SHAP contributions and shap_risk all agree")

[C] booster.num_boosted_rounds(): int
    meaning: trees in the model
    275
[C] margin: ndarray shape=(5,) float64 min=-1.0433 max=3.12782 nan=0
    meaning: eta(x): log hazard ratio, larger = earlier churn
    [-1.0433  1.8971  3.058   3.1278  3.1278]
[C] booster.predict(matrix): ndarray shape=(5,) float32 min=0.352289 max=22.8241 nan=0
    meaning: without output_margin XGBoost returns exp(margin)
    [ 0.3523  6.6668 21.2853 22.8241 22.8241]
[C] margin by number of trees: ndarray shape=(8, 5) float32 min=-1.56381 max=3.12782 nan=0
    meaning: rows = checkpoints, columns = players
    [[ 0.0292  0.0517  0.0517  0.0517  0.0517]
     [-0.0157  0.0966  0.0966  0.0966  0.0966]
     [-0.0718  0.1527  0.1527  0.1527  0.1527]
     [-0.2402  0.3211  0.3211  0.3211  0.3211]
     [-0.5209  0.6018  0.6018  0.6018  0.6018]]
[C] leaves: ndarray shape=(5, 275) float32 min=1 max=30 nan=0
    meaning: leaf index of every player in every tree: (players, trees)
    [[12. 24. 10. 24. 18. 14.]
     [

In [9]:
labels = [
    f"player {player_id} (recency {recency_days})"
    for player_id, recency_days in zip(debug["player_id"], debug["recency_days"], strict=True)
]
fig = go.Figure()
for column, label in enumerate(labels):
    color = qualitative.Plotly[column % len(qualitative.Plotly)]
    trace = go.Scatter(x=checkpoints, y=by_trees[:, column], name=label, line={"color": color})
    fig.add_trace(trace)
fig.update_layout(
    title="Stage C: how the margin builds up as trees are added",
    xaxis_title="number of trees used",
    yaxis_title=f"margin ({production.name})",
    template="plotly_white",
    height=380,
)
fig.show()

## Stage D. Margin to a survival curve

Both models turn one number per player into a whole curve `S(t | x)`, the probability of
not having churned by day `t`, on the grid of days 1..85. The result is a matrix with
one row per player and one column per day.

Churn takes 30 inactive days, so a player last active `r` days ago cannot churn during
the first `d = 30 - r` days (`churn_free_days`). Both models were trained on the time
left after that delay and are evaluated on the player's own clock `u = t - d`:
`S(t | x) = 1` while `u <= 0`, and the model formula with `u` in place of `t` after it.

### D1. Cox

`S(t | x) = exp(-H0(u) * exp(eta(x)))`. `H0` is the Breslow baseline cumulative hazard
estimated on the training data, on the model clock, and stored with the model as a step
function: its jump times and its value after each jump.

In [10]:
stage = "D1"
role = "production" if cox is production else "evaluation"
print(f"Cox model in use: the {role} model\n")

eta_cox = cox.predict_margin(X)
peek("eta", eta_cox, "log hazard ratio per player")
hazard_ratio = np.exp(eta_cox)
peek("hazard_ratio", hazard_ratio, "exp(eta): how many times the baseline hazard the player has")

baseline_times = np.asarray(cox.state_["baseline_times"])
baseline_hazard = np.asarray(cox.state_["baseline_hazard"])
peek("baseline_times", baseline_times, "days on which the training data had churn events")
peek("baseline_hazard", baseline_hazard, "H0 right after each of those days (non-decreasing)")

delay = X[config.DELAY_COLUMN].to_numpy(dtype=float)
peek("delay", delay, "days during which the player cannot churn: 30 - recency_days")
model_times = TIMES[None, :] - delay[:, None]
peek("model_times", model_times, "u = t - delay: each grid day on the player's clock", on_grid=True)
possible = model_times > 0
peek("possible", possible, "True where churn is possible by day t: (players, days)", on_grid=True)

step = np.searchsorted(baseline_times, model_times, side="right")
peek("step", step, "how many jumps of H0 have happened by u: (players, days)", on_grid=True)
H0 = np.concatenate([[0.0], baseline_hazard])[step]
peek("H0", H0, "baseline cumulative hazard at each player's u", on_grid=True)
cumulative_hazard = hazard_ratio[:, None] * H0
peek(
    "cumulative_hazard",
    cumulative_hazard,
    "H(t | x) = exp(eta) * H0(u): (players, days)",
    on_grid=True,
)
surv_cox = np.where(possible, np.exp(-cumulative_hazard), 1.0)
peek("S (Cox, raw)", surv_cox, "survival probability, row = player, column = day", on_grid=True)

assert np.allclose(surv_cox, cox.predict_survival(X, TIMES))
print("\nOK: equals CoxModel.predict_survival")

Cox model in use: the production model

[D1] eta: ndarray shape=(5,) float64 min=-1.0433 max=3.12782 nan=0
    meaning: log hazard ratio per player
    [-1.0433  1.8971  3.058   3.1278  3.1278]
[D1] hazard_ratio: ndarray shape=(5,) float64 min=0.352289 max=22.8241 nan=0
    meaning: exp(eta): how many times the baseline hazard the player has
    [ 0.3523  6.6668 21.2853 22.8241 22.8241]
[D1] baseline_times: ndarray shape=(74,) float64 min=1 max=78 nan=0
    meaning: days on which the training data had churn events
    [1. 2. 3. 4. 5. 6. 7. 8.]
[D1] baseline_hazard: ndarray shape=(74,) float64 min=0.0446708 max=0.280417 nan=0
    meaning: H0 right after each of those days (non-decreasing)
    [0.0447 0.0458 0.0479 0.0507 0.0541 0.0576 0.0613 0.0652]
[D1] delay: ndarray shape=(5,) float64 min=0 max=29 nan=0
    meaning: days during which the player cannot churn: 30 - recency_days
    [29. 22. 14.  7.  0.]
[D1] model_times: ndarray shape=(5, 85) float64 min=-28 max=85 nan=0
    meaning: u

### D2. AFT

`log U = mu(x) + sigma * Z` for the time `U` left after the delay, so
`S(t | x) = P(Z > (log u - mu(x)) / sigma)` for `u > 0`. The noise
distribution of `Z` and its scale `sigma` are hyperparameters stored with the model.

In [11]:
stage = "D2"
role = "production" if aft is production else "evaluation"
print(f"AFT model in use: the {role} model, noise = {aft.distribution}, sigma = {aft.scale:.4f}\n")

mu = aft.predict_margin(X)
peek("mu", mu, "location of log(days to churn) per player")
peek("exp(mu)", np.exp(mu), "the same on the scale of days after the delay")
# The placeholder 1.0 keeps the logarithm defined where churn is impossible.
log_u = np.log(np.where(possible, model_times, 1.0))
peek("log_u", log_u, "log of the days on the player's clock (0 where impossible)", on_grid=True)
z = (log_u - mu[:, None]) / aft.scale
peek(
    "z",
    z,
    "standardised distance of each day from the player's location: (players, days)",
    on_grid=True,
)

# Survival function of the standardised noise Z, one per XGBoost AFT distribution.
noise_survival = {
    "normal": lambda value: ndtr(-value),
    "logistic": lambda value: expit(-value),
    "extreme": lambda value: np.exp(-np.exp(value)),
}[aft.distribution]
surv_aft = np.where(possible, noise_survival(z), 1.0)
peek("S (AFT, raw)", surv_aft, "survival probability, row = player, column = day", on_grid=True)

assert np.allclose(surv_aft, aft.predict_survival(X, TIMES))
print("\nOK: equals AFTModel.predict_survival")

AFT model in use: the evaluation model, noise = logistic, sigma = 1.1411

[D2] mu: ndarray shape=(5,) float64 min=-0.110549 max=7.0192 nan=0
    meaning: location of log(days to churn) per player
    [ 7.0192  2.111   0.1037 -0.1105  0.0272]
[D2] exp(mu): ndarray shape=(5,) float64 min=0.895342 max=1117.89 nan=0
    meaning: the same on the scale of days after the delay
    [1117.8886    8.2567    1.1093    0.8953    1.0276]
[D2] log_u: ndarray shape=(5, 85) float64 min=0 max=4.44265 nan=0
    meaning: log of the days on the player's clock (0 where impossible)
    days 1, 7, 14, 30, 60, 85 (first rows):
    [[0.     0.     0.     0.     3.434  4.0254]
     [0.     0.     0.     2.0794 3.6376 4.1431]
     [0.     0.     0.     2.7726 3.8286 4.2627]
     [0.     0.     1.9459 3.1355 3.9703 4.3567]
     [0.     1.9459 2.6391 3.4012 4.0943 4.4427]]
[D2] z: ndarray shape=(5, 85) float64 min=-6.1515 max=3.91503 nan=0
    meaning: standardised distance of each day from the player's location: 

## Stage E. Calibration

The raw curve of the production model is recalibrated day by day. For every day `t` the
calibrator holds a monotone map (the knots of an isotonic regression fitted on the
snapshot of the calibrator's own model) from the raw churn probability `1 - S(t | x)` to
the observed frequency of churn by day `t`. After the 85 maps are applied, a running
maximum over the days makes the churn probability non-decreasing in time again. A raw
probability of exactly zero, which is what stage D gives before churn is possible, stays
zero.

In [12]:
stage = "E"
surv_raw = production.predict_survival(X, TIMES)
churn_raw = 1 - surv_raw
peek("churn_raw", churn_raw, "raw P(churn by day t), (players, days)", on_grid=True)

k = int(np.flatnonzero(TIMES == HORIZON)[0])
knots_x, knots_y = calibrator.knots_[k]
peek(f"knots_x (day {HORIZON})", knots_x, "raw probabilities at which the map changes")
peek(f"knots_y (day {HORIZON})", knots_y, "calibrated probabilities at those points")
one_day = np.interp(churn_raw[:, k], knots_x, knots_y)
peek(f"calibrated churn (day {HORIZON})", one_day, "the day-60 map applied to the players")

mapped = np.column_stack(
    [np.interp(churn_raw[:, day], x, y) for day, (x, y) in enumerate(calibrator.knots_)]
)
peek("mapped", mapped, "all 85 maps applied column by column", on_grid=True)
monotone = np.maximum.accumulate(mapped, axis=1)
peek(
    "monotone",
    monotone,
    "running maximum over days: churn probability never decreases",
    on_grid=True,
)
raised = int((monotone > mapped).sum())
print(f"    cells raised by the running maximum: {raised} of {mapped.size}")
surv = 1 - monotone
peek("S (calibrated)", surv, "final survival curves used for scoring", on_grid=True)

assert np.allclose(surv, calibrator.transform(surv_raw, TIMES))
print("\nOK: equals IsotonicSurvivalCalibrator.transform")

[E] churn_raw: ndarray shape=(5, 85) float64 min=0 max=0.998339 nan=0
    meaning: raw P(churn by day t), (players, days)
    days 1, 7, 14, 30, 60, 85 (first rows):
    [[0.     0.     0.     0.0156 0.0543 0.085 ]
     [0.     0.     0.     0.3525 0.7103 0.8358]
     [0.     0.     0.     0.8785 0.9898 0.9973]
     [0.     0.     0.7529 0.9458 0.9959 0.9983]
     [0.6393 0.7529 0.8741 0.9707 0.9977 0.9983]]
[E] knots_x (day 60): ndarray shape=(80,) float64 min=0.01672 max=0.997609 nan=0
    meaning: raw probabilities at which the map changes
    [0.0167 0.0285 0.0285 0.0352 0.0354 0.0393 0.0393 0.0517]
[E] knots_y (day 60): ndarray shape=(80,) float64 min=0 max=1 nan=0
    meaning: calibrated probabilities at those points
    [0.     0.     0.0211 0.0211 0.0308 0.0308 0.038  0.038 ]
[E] calibrated churn (day 60): ndarray shape=(5,) float64 min=0.0416667 max=1 nan=0
    meaning: the day-60 map applied to the players
    [0.0417 0.6393 0.9282 0.9825 1.    ]
[E] mapped: ndarray shape=(5,

Both kinds of model go through exactly this code, each with its own calibrator. The
60-day churn probability of the followed players, before and after calibration:

In [13]:
at_horizon = {"player_id": debug["player_id"].to_numpy()}
for name, model in (("cox", cox), ("aft", aft)):
    raw_curves = model.predict_survival(X, TIMES)
    calibrated_curves = calibrators[name].transform(raw_curves, TIMES)
    peek(
        f"S ({name}, calibrated)", calibrated_curves, f"calibrated curves of '{name}'", on_grid=True
    )
    at_horizon[f"{name} raw"] = 1 - raw_curves[:, k]
    at_horizon[f"{name} calibrated"] = 1 - calibrated_curves[:, k]
display(pd.DataFrame(at_horizon).set_index("player_id"))

[E] S (cox, calibrated): ndarray shape=(5, 85) float64 min=0 max=1 nan=0
    meaning: calibrated curves of 'cox'
    days 1, 7, 14, 30, 60, 85 (first rows):
    [[1.     1.     1.     1.     0.9582 0.9204]
     [1.     1.     1.     0.4603 0.3538 0.3   ]
     [1.     1.     1.     0.0886 0.0718 0.0519]
     [1.     1.     0.025  0.025  0.0161 0.    ]
     [0.     0.     0.     0.     0.     0.    ]]
[E] S (aft, calibrated): ndarray shape=(5, 85) float64 min=0 max=1 nan=0
    meaning: calibrated curves of 'aft'
    days 1, 7, 14, 30, 60, 85 (first rows):
    [[1.     1.     1.     1.     0.9678 0.919 ]
     [1.     1.     1.     0.4118 0.1774 0.1512]
     [1.     1.     1.     0.0592 0.0196 0.0123]
     [1.     1.     0.0396 0.0382 0.013  0.0123]
     [0.     0.     0.     0.     0.     0.    ]]


,cox raw,cox calibrated,aft raw,aft calibrated
player_id,,,,
2,0.0543,0.0418,0.0414,0.0322
121,0.7103,0.6462,0.7921,0.8226
1194,0.9898,0.9282,0.9632,0.9804
1064,0.9959,0.9839,0.9728,0.9870
438,0.9977,1.0000,0.9725,1.0000


In [14]:
days = [0, *TIMES]
fig = go.Figure()
for row, label in enumerate(labels):
    color = qualitative.Plotly[row % len(qualitative.Plotly)]
    style = {"legendgroup": label, "line": {"color": color}}
    fig.add_trace(go.Scatter(x=days, y=[1.0, *surv[row]], name=label, **style))
    style = {"legendgroup": label, "line": {"color": color, "dash": "dot"}, "showlegend": False}
    fig.add_trace(go.Scatter(x=days, y=[1.0, *surv_raw[row]], name=label, **style))
fig.add_vline(x=HORIZON, line_dash="dash", line_color="#6b7280")
fig.update_layout(
    title="Stage E: survival curves of the followed players, raw (dotted) and calibrated (solid)",
    xaxis_title="days since the snapshot",
    yaxis_title="probability of not having churned",
    yaxis_tickformat=".0%",
    template="plotly_white",
    height=440,
)
fig.show()

## Stage F. Curve to scorecard

Everything a scorecard shows is read off the final curve (calibrated or raw, as model
selection chose) and the SHAP contributions:

- churn probability within `h` days: `1 - S(h | x)`;
- remaining useful life: `E[min(T, 85)] = 1 + S(1) + ... + S(84)` days;
- risk score: the margin, signed so that larger means riskier once churn is possible. It
  ignores the delay, so it does not rank players by churn at a fixed day; ties in the
  probability are broken by the raw 60-day probability instead;
- risk segment: fixed thresholds on the 60-day probability;
- risk drivers: the three features with the largest positive SHAP contribution.

In [15]:
stage = "F"
final = surv if scoring_calibrator is not None else surv_raw
peek("final", final, f"curves behind the scores: {best_variant}", on_grid=True)
churn_prob = {h: 1 - final[:, TIMES == h][:, 0] for h in config.SCORE_HORIZONS}
for h, values in churn_prob.items():
    peek(f"churn_prob_{h}d", values, f"P(churn within {h} days)")
raw_churn_prob = 1 - surv_raw[:, k]
peek(f"raw_churn_prob_{HORIZON}d", raw_churn_prob, "uncalibrated P(churn): breaks ties in the rank")
rul = 1.0 + final[:, :-1].sum(axis=1)
peek("rul_days", rul, "expected days until churn, capped at 85")
risk_score = production.risk_sign * eta
peek("risk_score", risk_score, "margin signed so that larger = riskier")

names, bounds = zip(*config.RISK_SEGMENTS, strict=True)
segment = np.select([churn_prob[HORIZON] >= bound for bound in bounds], names, default=names[-1])
peek(
    "risk_segment",
    segment,
    f"thresholds on the {HORIZON}-day probability: {dict(config.RISK_SEGMENTS)}",
)

top = np.argsort(-shap, axis=1)[:, :3]
peek("top", top, "column indices of the three largest SHAP contributions per player")
drivers = np.array(FEATURES)[top]
peek("drivers", drivers, "the same as feature names")
driver_shap = np.take_along_axis(shap, top, axis=1)
peek("driver_shap", driver_shap, "their SHAP values on the risk-score scale")

[F] final: ndarray shape=(5, 85) float64 min=0 max=1 nan=0
    meaning: curves behind the scores: calibrated
    days 1, 7, 14, 30, 60, 85 (first rows):
    [[1.     1.     1.     1.     0.9582 0.9204]
     [1.     1.     1.     0.4603 0.3538 0.3   ]
     [1.     1.     1.     0.0886 0.0718 0.0519]
     [1.     1.     0.025  0.025  0.0161 0.    ]
     [0.     0.     0.     0.     0.     0.    ]]
[F] churn_prob_7d: ndarray shape=(5,) float64 min=0 max=1 nan=0
    meaning: P(churn within 7 days)
    [0. 0. 0. 0. 1.]
[F] churn_prob_14d: ndarray shape=(5,) float64 min=0 max=1 nan=0
    meaning: P(churn within 14 days)
    [0.    0.    0.    0.975 1.   ]
[F] churn_prob_30d: ndarray shape=(5,) float64 min=0 max=1 nan=0
    meaning: P(churn within 30 days)
    [0.     0.5397 0.9114 0.975  1.    ]
[F] churn_prob_60d: ndarray shape=(5,) float64 min=0.0418251 max=1 nan=0
    meaning: P(churn within 60 days)
    [0.0418 0.6462 0.9282 0.9839 1.    ]
[F] raw_churn_prob_60d: ndarray shape=(5,) float

The library does all of stages B-F in one call, `score_players`, and sorts the players
from the riskiest down. Its output must equal the hand-made numbers above. For the
scoring snapshot it must also equal what the pipeline saved in `artifacts/scores.parquet`
(there the rank is the place among all players at risk, here among the followed ones).

In [16]:
scores, curves = score_players(production, scoring_calibrator, debug)
library = scores.set_index("player_id").loc[debug["player_id"]]
for h, values in churn_prob.items():
    assert np.allclose(library[f"churn_prob_{h}d"], values)
assert np.allclose(library[f"raw_churn_prob_{HORIZON}d"], raw_churn_prob)
assert np.allclose(library["rul_days"], rul)
assert np.allclose(library["risk_score"], risk_score)
assert (library["risk_segment"].to_numpy() == segment).all()
assert (library[["driver_1", "driver_2", "driver_3"]].to_numpy() == drivers).all()
print("OK: equals pipeline.score_players")

if SNAPSHOT == config.SCORING_SNAPSHOT:
    saved = pd.read_parquet(ARTIFACTS_DIR / "scores.parquet").set_index("player_id")
    saved = saved.loc[debug["player_id"]]
    for h, values in churn_prob.items():
        assert np.allclose(saved[f"churn_prob_{h}d"], values)
    assert np.allclose(saved["rul_days"], rul)
    print("OK: equals artifacts/scores.parquet for these players")

shown = ["risk_rank", "recency_days", *[f"churn_prob_{h}d" for h in config.SCORE_HORIZONS]]
shown += ["rul_days", "risk_score", "risk_segment", "driver_1", "driver_2", "driver_3"]
display(scores.set_index("player_id")[shown])

OK: equals pipeline.score_players
OK: equals artifacts/scores.parquet for these players


,risk_rank,recency_days,churn_prob_7d,churn_prob_14d,churn_prob_30d,churn_prob_60d,rul_days,risk_score,risk_segment,driver_1,driver_2,driver_3
player_id,,,,,,,,,,,,
438,1,30,1.0,1.000,1.0000,1.0000,1.0000,3.1278,high,recency_days,active_days_7d,activity_ratio_7d_30d
1064,2,23,0.0,0.975,0.9750,0.9839,9.3530,3.1278,high,recency_days,active_days_7d,activity_ratio_7d_30d
1194,3,16,0.0,0.000,0.9114,0.9282,20.3516,3.0580,high,recency_days,active_days_7d,activity_ratio_7d_30d
121,4,8,0.0,0.000,0.5397,0.6462,46.4678,1.8971,medium,active_days_7d,activity_ratio_7d_30d,acquisition_channel
2,5,1,0.0,0.000,0.0000,0.0418,82.7239,-1.0433,low,acquisition_channel,ggr_30d,deposits_30d


## Stage G. Check against the churn definition

Churn is 30 inactive days in a row, so a player last active `r` days before the snapshot
cannot churn before day `31 - r`. Stage D builds this into the curves, and the
calibrator maps a probability of zero to zero. The check below measures how much churn
probability a curve places on days when churn is impossible for that player. It must be
exactly zero for every player at risk, raw and calibrated.

The last chart shows the other side of the bound: the first day on which a curve leaves
100%, by `recency_days`, against the earliest possible day `31 - r`.

In [17]:
stage = "G"


def impossible_mass(curves: np.ndarray, earliest_day: np.ndarray) -> np.ndarray:
    """Predicted probability of churning before the earliest possible day."""
    last_impossible = earliest_day - 1  # 0 = churn is possible from day 1
    column = np.maximum(last_impossible, 1) - 1
    mass = 1 - curves[np.arange(len(curves)), column]
    return np.where(last_impossible >= 1, mass, 0.0)


earliest = (config.INACTIVITY_DAYS + 1 - X["recency_days"]).to_numpy()
peek("earliest", earliest, "first day on which each followed player can churn: 31 - recency")
check = pd.DataFrame(
    {
        "player_id": debug["player_id"].to_numpy(),
        "recency_days": X["recency_days"].to_numpy(),
        "earliest_possible_day": earliest,
        "raw P(churn before it)": impossible_mass(surv_raw, earliest),
        "calibrated P(churn before it)": impossible_mass(surv, earliest),
    }
)
display(check)

X_all = frame[MODEL_COLUMNS]
raw_all = production.predict_survival(X_all, TIMES)
surv_all = calibrator.transform(raw_all, TIMES)
peek("raw_all", raw_all, "raw curves of every player at risk on the snapshot", on_grid=True)
peek("surv_all", surv_all, "the same curves after calibration", on_grid=True)
earliest_all = (config.INACTIVITY_DAYS + 1 - X_all["recency_days"]).to_numpy()
mass_all = {
    "raw": impossible_mass(raw_all, earliest_all),
    "calibrated": impossible_mass(surv_all, earliest_all),
}
summary = pd.DataFrame(
    {
        name: {"mean": mass.mean(), "max": mass.max(), "players above 0": int((mass > 0).sum())}
        for name, mass in mass_all.items()
    }
)
print(f"\nProbability placed on impossible days, {len(frame):,} players at risk:")
display(summary)
assert all((mass == 0).all() for mass in mass_all.values())
print("OK: no curve places any probability on a day when churn is impossible")

[G] earliest: ndarray shape=(5,) int64 min=1 max=30 nan=0
    meaning: first day on which each followed player can churn: 31 - recency
    [30 23 15  8  1]


,player_id,recency_days,earliest_possible_day,raw P(churn before it),calibrated P(churn before it)
0,2,1,30,0.0,0.0
1,121,8,23,0.0,0.0
2,1194,16,15,0.0,0.0
3,1064,23,8,0.0,0.0
4,438,30,1,0.0,0.0


[G] raw_all: ndarray shape=(4328, 85) float64 min=0.00166108 max=1 nan=0
    meaning: raw curves of every player at risk on the snapshot
    days 1, 7, 14, 30, 60, 85 (first rows):
    [[1.     1.     1.     0.9844 0.9457 0.915 ]
     [1.     1.     1.     0.9599 0.8651 0.7958]
     [1.     1.     1.     0.9742 0.9115 0.8629]
     [1.     1.     1.     0.2479 0.0366 0.0115]
     [1.     1.     1.     0.9626 0.8755 0.8128]]
[G] surv_all: ndarray shape=(4328, 85) float64 min=0 max=1 nan=0
    meaning: the same curves after calibration
    days 1, 7, 14, 30, 60, 85 (first rows):
    [[1.     1.     1.     1.     0.9582 0.9204]
     [1.     1.     1.     0.959  0.8404 0.7522]
     [1.     1.     1.     0.9931 0.9056 0.8529]
     [1.     1.     1.     0.0886 0.0718 0.0621]
     [1.     1.     1.     0.9469 0.8404 0.7628]]



Probability placed on impossible days, 4,328 players at risk:


,raw,calibrated
mean,0.0,0.0
max,0.0,0.0
players above 0,0.0,0.0


OK: no curve places any probability on a day when churn is impossible


In [18]:
falls = raw_all < 1
first_fall = pd.Series(TIMES[np.argmax(falls, axis=1)]).where(falls.any(axis=1))
by_recency = first_fall.groupby(X_all["recency_days"].to_numpy()).agg(["min", "size"])
peek("by_recency", by_recency, "first day on which a raw curve is below 100%, by recency_days")
earliest_by_recency = config.INACTIVITY_DAYS + 1 - by_recency.index
assert (by_recency["min"] >= earliest_by_recency).all()

bars = go.Bar(x=by_recency.index, y=by_recency["min"], customdata=by_recency["size"])
bars.name = "first day below 100%"
bars.hovertemplate = "recency %{x} days<br>day %{y}<br>%{customdata} players<extra></extra>"
bound = go.Scatter(x=by_recency.index, y=earliest_by_recency, name="earliest possible day")
bound.line = {"color": "#dc2626", "dash": "dash"}
fig = go.Figure([bars, bound])
fig.update_layout(
    title="Stage G: the first day on which a curve leaves 100%, by recency",
    xaxis_title="recency_days (days since the last activity)",
    yaxis_title="day since the snapshot",
    template="plotly_white",
    height=360,
)
fig.show()

[G] by_recency: DataFrame shape=(30, 2) mixed
    meaning: first day on which a raw curve is below 100%, by recency_days
       min  size
    1   30  2120
    2   29   772
    3   28   444


## Journal

Every object shown above, in order: where it appears, its shape and what it is.

In [19]:
display(pd.DataFrame(JOURNAL))

,stage,object,type,shape,dtype,meaning
0,A,players,DataFrame,"(20000, 5)",mixed,one row per player: registration date and static traits
1,A,activity,DataFrame,"(1700671, 9)",mixed,one row per player and active day
2,A,churn_dates,Series,"(15672,)",datetime64[us],date of the first churn by player_id; absent = never churned
3,A,history,DataFrame,"(1700671, 9)",mixed,activity strictly before the snapshot: all the model may see
4,A,last_active,Series,"(20000,)",datetime64[us],last active day of every player who has played
5,A,still_at_risk,ndarray,"(20000,)",bool,True if the first churn has not happened before the snapshot
6,A,at_risk,Index,"(4328,)",int64,player ids that get a row and a prediction
7,A,window,DataFrame,"(89675, 9)",mixed,"activity of the last 30 days, the source of the *_30d features"
8,A,recency_days,Series,"(4328,)",int64,"days since the last activity, 1 = played yesterday"
9,A,frame,DataFrame,"(4328, 19)",mixed,"output of snapshot_features: ids, date, 15 features, delay, churn date"
